<h1><center>Stock Analysis</center></h1>

Import necessary libraries

In [109]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
from alpha_vantage.timeseries import TimeSeries
from dotenv import load_dotenv
import os
import datetime


Import API and retrieve data

In [110]:
load_dotenv()

True

In [111]:
# Imports data
ts = TimeSeries(os.getenv('api_key'), output_format='pandas')

df = ts.get_monthly_adjusted('QBTS')

stock = df[0] 

In [112]:
# Converts data into a pandas dataframe
stock = pd.DataFrame(stock)

In [113]:
# View first five rows
stock.head()

,1. open,2. high,3. low,4. close,5. adjusted close,6. volume,7. dividend amount
date,,,,,,,
2025-10-29,24.390,46.75,23.91,34.25,34.25,1.444134e+09,0.0
2025-09-30,15.155,29.18,14.78,24.71,24.71,1.144078e+09,0.0
2025-08-29,16.510,19.17,14.20,15.62,15.62,7.816219e+08,0.0
2025-07-31,15.200,20.56,14.29,17.19,17.19,1.006192e+09,0.0
2025-06-30,16.235,18.95,13.57,14.64,14.64,1.087715e+09,0.0


In [114]:
# View colmns
stock.columns

Index(['1. open', '2. high', '3. low', '4. close', '5. adjusted close',
       '6. volume', '7. dividend amount'],
      dtype='object')

In [115]:
# Rename columns
stock.rename(columns={'1. open': 'open', '2. high': 'high', '3. low': 'low', 
'4. close': 'close', '5. adjusted close': 'adj_close',
'6. volume': 'volume', '7. dividend amount': 'div_amt'}, inplace = True)

In [116]:
# view first 5 rows to ensure column names changed.
stock.head()

,open,high,low,close,adj_close,volume,div_amt
date,,,,,,,
2025-10-29,24.390,46.75,23.91,34.25,34.25,1.444134e+09,0.0
2025-09-30,15.155,29.18,14.78,24.71,24.71,1.144078e+09,0.0
2025-08-29,16.510,19.17,14.20,15.62,15.62,7.816219e+08,0.0
2025-07-31,15.200,20.56,14.29,17.19,17.19,1.006192e+09,0.0
2025-06-30,16.235,18.95,13.57,14.64,14.64,1.087715e+09,0.0


<h1><center>EDA</center><h1>

In [117]:
stock.dtypes

open         float64
high         float64
low          float64
close        float64
adj_close    float64
volume       float64
div_amt      float64
dtype: object

In [118]:
# drop unnecessary column
stock.drop(columns=('div_amt'), inplace=True)

In [119]:
stock['pct_ch_adj'] = stock['adj_close'].pct_change() * 100

In [120]:
stock['pct_ch_adj'] = stock['pct_ch_adj'].round(2)

In [121]:
stock.head()

,open,high,low,close,adj_close,volume,pct_ch_adj
date,,,,,,,
2025-10-29,24.390,46.75,23.91,34.25,34.25,1.444134e+09,NaN
2025-09-30,15.155,29.18,14.78,24.71,24.71,1.144078e+09,-27.85
2025-08-29,16.510,19.17,14.20,15.62,15.62,7.816219e+08,-36.79
2025-07-31,15.200,20.56,14.29,17.19,17.19,1.006192e+09,10.05
2025-06-30,16.235,18.95,13.57,14.64,14.64,1.087715e+09,-14.83


In [122]:
# Function to calculate Sharpe Ratio.
def calculate_sharp_ratio(ret, ann_rfr, periods):
    daily_rfr = ann_rfr/periods
    excess_ret = ret-daily_rfr
    mean_ex_ret = excess_ret.mean()
    std_ex_ret = excess_ret.std()

    if std_ex_ret == 0:
        return np.nan

    sharp_ratio = (mean_ex_ret/std_ex_ret) * np.sqrt(periods)

    return sharp_ratio

In [125]:
# call function to calculate Sharpe Ratio.
ann_rfr = 0.377
periods = 12
ret = stock['pct_ch_adj']
sharpe = calculate_sharp_ratio(ret, ann_rfr, periods).round(2)
print(f"The Sharpe ratio for QBTS is:\n{sharpe}")

The Sharpe ratio for QBTS is:
0.43
